In [ ]:
from datasets import load_dataset
import pandas as pd
from tqdm.auto import tqdm
import fasttext
from pathlib import Path
import re

print("ready")


In [1]:
from datasets import load_dataset
import pandas as pd
from itertools import islice
from pathlib import Path

N_SCAN = 1000
N_SAMPLE = 10

propella = load_dataset(
    "openeurollm/propella-annotations",
    "finepdfs",
    split="swe_Latn",
    streaming=True,
)

finepdfs = load_dataset(
    "HuggingFaceFW/finepdfs",
    "swe_Latn",
    split="train",
    streaming=True,
)

propella_rows = []
for row in islice(propella, N_SCAN):
    propella_rows.append({
        "id": row["id"],
        "one_sentence_description": row["one_sentence_description"],
        "content_quality": row["content_quality"],
        "educational_value": row["educational_value"],
        "content_safety": row["content_safety"],
        "pii_presence": row["pii_presence"],
    })

finepdf_rows = []
for row in islice(finepdfs, N_SCAN):
    finepdf_rows.append({
        "id": row["id"],
        "raw_text": row["text"],
        "language": row["language"],
        "url": row["url"],
    })

propella_df = pd.DataFrame(propella_rows)
finepdf_df = pd.DataFrame(finepdf_rows)

merged = propella_df.merge(finepdf_df, on="id", how="inner")

sample = merged.sample(
    n=min(N_SAMPLE, len(merged)),
    random_state=42,
).copy()

sample["raw_text_excerpt"] = sample["raw_text"].str.slice(0, 5000)

output = sample[[
    "id",
    "language",
    "one_sentence_description",
    "raw_text_excerpt",
    "content_quality",
    "educational_value",
    "content_safety",
    "pii_presence",
    "url",
]]

Path("../data/processed").mkdir(parents=True, exist_ok=True)

output.to_csv(
    "../data/processed/human_test_raw_text_and_descriptions_10.csv",
    index=False,
)

print("Matched rows:", len(merged))
print("Saved rows:", len(output))
output


README.md: 0.00B [00:00, ?B/s]

Matched rows: 932
Saved rows: 10


,id,language,one_sentence_description,raw_text_excerpt,content_quality,educational_value,content_safety,pii_presence,url
829,<urn:uuid:d5609ca4-7ae7-42fa-b4cd-b9f285db723a>,swe_Latn,Technical product brochure describing the feat...,Kubus – Ljus utanför boxen\n\nFasadbelysning o...,good,basic,safe,no_pii,https://www.erco.com/cdn/downloaddata/2021/db/...
70,<urn:uuid:e2eb457f-fd41-437f-8469-6f53d30a0b2d>,swe_Latn,Monthly report for the Lannebo Mixfond mutual ...,Lannebo Mixfond\n\nSverigeregistrerad blandfon...,good,basic,safe,no_pii,https://www.lannebofonder.se/upl/files/97946.pdf
631,<urn:uuid:87517814-05a2-40a8-9e51-e4360e676cf8>,swe_Latn,Official municipal decision rejecting a citize...,2021-01-27\n\nUpplevelseförvaltningen Klara Ha...,good,none,safe,contains_pii,https://yh.enkoping.se/download/18.78952cc1774...
506,<urn:uuid:4def5d5a-5b45-428c-a8aa-43d136a742c3>,swe_Latn,Official minutes and decisions from the Botkyr...,Arbetsmarknads- och vuxenutbildningsnämnden\n\...,good,none,safe,no_pii,https://www.botkyrka.se/download/18.322f2bf016...
703,<urn:uuid:fc17b5d7-caa0-4769-8bda-1fd9aaa34fca>,swe_Latn,Instructional guide explaining the concepts of...,Mappar och filer\n\nViktiga papper sätter vi i...,good,high,safe,contains_pii,https://kungsholmen.seniornet.se/wp-content/up...
96,<urn:uuid:b0216d1b-23c8-4149-b0a3-3e1812cb3b93>,swe_Latn,Promotional text for the NACREO MAN hair colo...,Powered by TCPDF (www.tcpdf.org)\n\nMan- nings...,poor,none,safe,no_pii,http://beautybazar.com/sv/man-ningsmedel-c-20/...
465,<urn:uuid:6b0d90f8-6b6a-4a78-8cf9-22942b9daa54>,swe_Latn,A Swedish parliamentary motion proposing to ab...,Motion till riksdagen 2020/21:2318\n\nav Boria...,good,basic,safe,no_pii,https://data.riksdagen.se/fil/0E5BF3E3-D20D-4E...
86,<urn:uuid:b0dd256d-9a2c-4e47-8b20-ffc8fbb682b5>,swe_Latn,Application form for a student field internshi...,ANSÖKAN OM MFS-STIPENDIUM\n\nPersonuppgifter\n...,good,none,safe,no_pii,https://www.hkr.se/globalassets/avdelningar/ex...
530,<urn:uuid:6148b8a1-7c57-4d72-ad7f-61b5a40f0b4e>,swe_Latn,Agenda for the 2016 annual meeting of the BMW ...,"BMW Club Schweden\n\nBMW Club Schweden, Årsmöt...",good,none,safe,no_pii,http://bmwcs.com/files/dokument/Dagordning%202...
350,<urn:uuid:67d59c8c-5578-455c-aa64-d31936a54348>,swe_Latn,A Swedish government committee directive from ...,Kommittédirektiv\n\nEtt stärkt arbete för anpa...,excellent,minimal,safe,no_pii,https://www.regeringen.se/contentassets/0b552c...


In [ ]:
propella = load_dataset(
    "openeurollm/propella-annotations",
    "finepdfs",
    split="swe_Latn",
    streaming=True,
)

first_propella = next(iter(propella))

print(first_propella.keys())
first_propella


In [ ]:
finepdfs = load_dataset(
    "HuggingFaceFW/finepdfs",
    "swe_Latn",
    split="train",
    streaming=True,
)

first_finepdf = next(iter(finepdfs))

print(first_finepdf.keys())
first_finepdf


In [ ]:
from datasets import load_dataset
import pandas as pd
from itertools import islice
from pathlib import Path

N = 100

propella = load_dataset(
    "openeurollm/propella-annotations",
    "finepdfs",
    split="swe_Latn",
    streaming=True,
)

finepdfs = load_dataset(
    "HuggingFaceFW/finepdfs",
    "swe_Latn",
    split="train",
    streaming=True,
)

propella_rows = []
for row in islice(propella, N):
    propella_rows.append({
        "id": row["id"],
        "content_quality": row["content_quality"],
        "information_density": row["information_density"],
        "educational_value": row["educational_value"],
        "content_safety": row["content_safety"],
        "pii_presence": row["pii_presence"],
        "content_integrity": row["content_integrity"],
        "content_ratio": row["content_ratio"],
        "content_length": row["content_length"],
    })

finepdf_rows = []
for row in islice(finepdfs, N):
    finepdf_rows.append({
        "id": row["id"],
        "text": row["text"],
        "language": row["language"],
        "full_doc_lid": row["full_doc_lid"],
        "full_doc_lid_score": row["full_doc_lid_score"],
        "page_average_lid": row["page_average_lid"],
        "page_average_lid_score": row["page_average_lid_score"],
        "token_count": row["token_count"],
        "minhash_cluster_size": row["minhash_cluster_size"],
        "duplicate_count": row["duplicate_count"],
        "url": row["url"],
    })

propella_df = pd.DataFrame(propella_rows)
finepdf_df = pd.DataFrame(finepdf_rows)

merged = propella_df.merge(finepdf_df, on="id", how="inner")

print("Propella rows:", len(propella_df))
print("FinePDFs rows:", len(finepdf_df))
print("Merged rows:", len(merged))

merged.head()


In [ ]:
merged["language_match_finepdfs"] = merged["language"] == merged["full_doc_lid"]
merged["low_confidence_finepdfs"] = merged["full_doc_lid_score"] < 0.70

Path("../data/processed").mkdir(parents=True, exist_ok=True)

language_features = merged[[
    "id",
    "language",
    "full_doc_lid",
    "full_doc_lid_score",
    "page_average_lid",
    "page_average_lid_score",
    "language_match_finepdfs",
    "low_confidence_finepdfs",
]]

language_features.to_csv(
    "../data/processed/language_confidence_finepdfs_swe_sample_55.csv",
    index=False,
)

merged.to_parquet(
    "../data/processed/merged_propella_finepdfs_swe_sample_55.parquet",
    index=False,
)

print("Saved language features:", len(language_features))
language_features.head()


In [ ]:
import sys
!{sys.executable} -m pip install datasets pandas tqdm fasttext-wheel ipykernel pyarrow


In [ ]:
import sys
print(sys.executable)
